# Machine Learning - Practical 07: Clustering and PCA

**Course:** Machine Learning - National University of Kyiv-Mohyla Academy (NaUKMA)

**Instructor:** Dmytro Kuzmenko - kuzmenko@ukma.edu.ua

| | |
|---|---|
| Week | 7 |
| Module | 7. Finding Structure Without Labels |
| Format | Practical session (not graded - exam preparation) |
| Estimated time | 1.5-2 h of active work including discussion |
| Prerequisites | P01-P05 |


## AI Use Disclosure

Fill this in before submitting (see course policy).

| Field | Your entry |
|---|---|
| AI tools used | |
| Nature of assistance | |
| Representative prompts or relevant interaction | |
| What I independently verified or changed | |

## Learning Objectives

- Explain why k-means is sensitive to feature scales and demonstrate the effect empirically.
- Choose the number of clusters with inertia and silhouette scores, and interpret both.
- Use PCA for dimensionality reduction: scree plot, cumulative variance, projection, and component interpretation.
- Sanity-check clustering quality with silhouette (unsupervised) versus ground-truth labels (supervised), and explain why k-means fails on ring-shaped data.

## Warm-up (10 min)

### Question 1 (multiple choice)

k-means found 3 clusters with silhouette 0.55. Which statement is true?

- A. Silhouette above 0.5 guarantees the clusters are meaningful in the real world.
- B. Silhouette measures how compact and separated the found clusters are; whether they correspond to real categories requires further analysis (stability, domain knowledge, downstream value).
- C. Silhouette can only be computed when ground-truth labels exist.
- D. Silhouette is meaningless unless the data were standardized.


**Your answer:**

### Question 2 (quick reasoning)

Features are measured in different units (age in years, income in thousands, count in pieces). Why does this matter for k-means but not for a decision tree?


**Your answer:**

### Question 3 (mini-interpretation)

PCA on a 64-dimensional dataset: PC1 explains 30% of variance, PC2 explains 28%. Is that a problem, and what does it tell you about the data?


**Your answer:**

## Guided Exercise (60 min)

We study clustering quality on synthetic data and PCA on the digits dataset. All experiments are deterministic.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_digits, make_circles
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
%matplotlib inline
np.random.seed(42)

### Task 1: k-means and feature scaling

**Experiment.** Two clusters are generated so that the separating signal lives only in the second feature (small scale), while the first feature is pure noise with a much larger scale. Run k-means (k=2) on the raw features and on standardized features, and compare the assignments against the true cluster ids with the adjusted Rand index (ARI).

In [2]:
rng = np.random.RandomState(42)
n = 300
true_labels = np.array([0] * 150 + [1] * 150)
noise_feature = rng.normal(0, 3.0, n)                     # large scale, no signal
signal_feature = np.where(true_labels == 1, 2.0, -2.0) + rng.normal(0, 0.6, n)
X_blobs = np.column_stack([noise_feature, signal_feature])

for name, Xd in [("raw", X_blobs),
                 ("standardized", StandardScaler().fit_transform(X_blobs))]:
    km = KMeans(n_clusters=2, n_init=10, random_state=42).fit(Xd)
    print("{}: silhouette = {:.3f}   ARI vs true labels = {:.3f}".format(
        name, silhouette_score(Xd, km.labels_), adjusted_rand_score(true_labels, km.labels_)))

raw: silhouette = 0.368   ARI vs true labels = 0.007
standardized: silhouette = 0.483   ARI vs true labels = 1.000


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
km_raw = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_blobs)
X_std = StandardScaler().fit_transform(X_blobs)
km_std = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_std)
for ax, Xd, km, title in [(axes[0], X_blobs, km_raw, "raw features"),
                          (axes[1], X_std, km_std, "standardized features")]:
    ax.scatter(Xd[:, 0], Xd[:, 1], c=km.labels_, cmap="bwr", s=12)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], marker="x",
               s=120, color="black", label="centroids")
    ax.set_title(title)
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
    ax.legend()
plt.tight_layout()

**Decision.** Which preprocessing would you apply before k-means in general, and why?


**Your answer:**

**Interpretation.** On the raw features the clusters found by k-means look "reasonable" by eye (two groups along feature 1), yet ARI is near zero. What does ARI tell you that the plot does not?


**Your answer:**

**Counterfactual.** Predict what happens to the raw-features result if the noise feature's scale is increased from 3.0 to 30.0. What happens to the standardized result?


**Your answer:**

### Task 2: Choosing k with inertia and silhouette

**Experiment.** Generate three well-separated blobs, standardize them, and run k-means for k = 2..6. Plot the inertia (within-cluster sum of squares) and the silhouette score for each k.

In [4]:
X3, y3 = make_blobs(n_samples=450, centers=3, cluster_std=1.1, random_state=42)
X3 = StandardScaler().fit_transform(X3)

ks = range(2, 7)
inertias, sil_scores = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X3)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X3, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(list(ks), inertias, marker="o")
axes[0].set_xlabel("k"); axes[0].set_ylabel("inertia"); axes[0].set_title("Elbow plot")
axes[1].plot(list(ks), sil_scores, marker="o")
axes[1].set_xlabel("k"); axes[1].set_ylabel("silhouette"); axes[1].set_title("Silhouette vs k")
plt.tight_layout()
print(pd.DataFrame({"k": list(ks), "inertia": np.round(inertias, 1),
                    "silhouette": np.round(sil_scores, 3)}).to_string(index=False))

 k  inertia  silhouette
 2    287.3       0.679
 3     34.1       0.829
 4     29.3       0.676
 5     24.8       0.502
 6     20.4       0.352


**Decision.** Which k would you choose from each plot, and do the two plots agree?


**Your answer:**

**Interpretation.** Inertia always decreases with k; why can it still be used, and what does silhouette add that inertia cannot show?


**Your answer:**

**Counterfactual.** Predict how both plots change if the three blobs heavily overlap. Which diagnostic fails first, and why?


**Your answer:**

### Task 3: PCA on the digits dataset

**Experiment.** Load the 8x8 digits (64 features), standardize, fit PCA, and examine: (1) the scree plot and cumulative variance with the 95% threshold, (2) the 2-D projection colored by digit, (3) the first principal components as 8x8 images.

In [5]:
digits = load_digits()
X_dig = StandardScaler().fit_transform(digits.data)
y_dig = digits.target
print("digits shape:", digits.data.shape)

pca = PCA().fit(X_dig)
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)
n95 = int(np.argmax(cum >= 0.95) + 1)
print("components needed for 95% variance:", n95)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(range(1, 21), evr[:20])
axes[0].set_xlabel("principal component"); axes[0].set_ylabel("explained variance ratio")
axes[0].set_title("Scree plot (first 20 PCs)")
axes[1].plot(range(1, len(cum) + 1), cum, marker=".", ms=3)
axes[1].axhline(0.95, color="red", ls="--", lw=1)
axes[1].set_xlabel("number of components"); axes[1].set_ylabel("cumulative variance")
axes[1].set_title("Cumulative explained variance")
plt.tight_layout()

digits shape: (1797, 64)
components needed for 95% variance: 40


In [6]:
proj = PCA(n_components=2, random_state=42).fit_transform(X_dig)
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(proj[:, 0], proj[:, 1], c=y_dig, cmap="tab10", s=14, alpha=0.7)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Digits projected onto the first two principal components")
plt.colorbar(sc, ticks=range(10), label="digit")
plt.tight_layout()

In [7]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.2))
for i, ax in enumerate(axes.ravel()):
    comp = pca.components_[i].reshape(8, 8)
    im = ax.imshow(comp, cmap="RdBu_r", vmin=-0.35, vmax=0.35)
    ax.set_title("PC{}".format(i + 1))
    ax.axis("off")
fig.suptitle("First 10 principal components of digits (as 8x8 images)")
plt.tight_layout()

**Interpretation.** Look at the eigendigits (PC images): what kind of variation does PC1 capture compared with, say, PC4? How does that explain the shape of the 2-D projection?


**Your answer:**

**Decision.** How many components would you keep, and which criterion (elbow, 95% variance, downstream accuracy) would you rely on for a real task?


**Your answer:**

### Task 4: Is clustering "working"? Silhouette vs ground truth

**Experiment.** On the standardized blobs, compute for k = 2..6 both the silhouette score (unsupervised - uses only the found clusters) and ARI against the true cluster ids (supervised - possible only because we generated the data).

In [8]:
ks = range(2, 7)
rows = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X3)
    rows.append({"k": k,
                 "silhouette": round(silhouette_score(X3, km.labels_), 3),
                 "ARI (vs true labels)": round(adjusted_rand_score(y3, km.labels_), 3)})
print(pd.DataFrame(rows).to_string(index=False))

 k  silhouette  ARI (vs true labels)
 2       0.679                 0.570
 3       0.829                 1.000
 4       0.676                 0.875
 5       0.502                 0.740
 6       0.352                 0.581


**Interpretation.** Both diagnostics peak at k=3 here. In a real dataset you have no true labels: which of the two would you use, and what would you do to build confidence in the clusters (stability, domain knowledge, downstream use)?


**Your answer:**

**Counterfactual.** Silhouette sometimes prefers k=2 for elongated or nested structures. Predict how silhouette would behave on two elongated parallel clusters, and why.


**Your answer:**

## Discussion Questions (15 min)

1. When would a high silhouette still mislead you? Give a concrete data shape.
2. k-means uses random initialization. What do n_init and random_state control, and why do results differ between runs without a fixed seed?
3. PCA is often applied before k-means. What does that change: the distances, the number of dimensions, or both? When is it a bad idea?
4. Is PCA "clustering"? Contrast what PCA finds (directions of maximal variance) with what k-means finds (groups of similar points).
5. A cluster is "real" - how would you argue for that claim without labels? Which evidence is necessary and which is sufficient?
6. Scaling before PCA and before k-means: same rule? What would change if one feature is measured in millimeters and another in kilometers?

## Challenge (25 min)

### Task 5: Why does k-means fail on rings?

**Experiment.** Generate two concentric rings (make_circles) and run k-means with k=2. Visualize the assignments and compute silhouette and ARI.

In [9]:
X_c, y_c = make_circles(n_samples=300, factor=0.5, noise=0.05, random_state=42)
km_c = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_c)
print("silhouette:", round(silhouette_score(X_c, km_c.labels_), 3))
print("ARI vs true rings:", round(adjusted_rand_score(y_c, km_c.labels_), 3))

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X_c[:, 0], X_c[:, 1], c=km_c.labels_, cmap="bwr", s=16)
ax.set_title("k-means (k=2) on concentric rings")
plt.tight_layout()

silhouette: 0.353
ARI vs true rings: -0.003


**Interpretation.** Explain, in terms of centroids and Euclidean distance, why k-means cannot recover the rings: what partition does it find instead, and what assumption about cluster shape does that reflect?


**Your answer:**

**Optional experiment (DBSCAN).** Run DBSCAN (try eps around 0.2, min_samples=5) on the rings and compare ARI with the k-means result. Why can a density-based method succeed here?


In [10]:
# Your code

## Takeaways

- k-means minimizes within-cluster squared distances; it finds convex, roughly spherical clusters and is blind to non-convex shapes such as rings.
- Feature scales decide what "distance" means: standardize before k-means and PCA unless you have a deliberate reason not to.
- Inertia always decreases with k; silhouette balances compactness and separation and is the better unsupervised diagnostic for choosing k.
- PCA is an unsupervised linear dimensionality reduction: use the scree plot and cumulative variance (e.g., 95%) to choose the number of components, and interpret components as images/weights.
- Clustering quality has two views: unsupervised metrics (silhouette, stability) and, when available, supervised sanity checks (ARI against labels). High silhouette does not imply real categories.
- DBSCAN clusters by density and handles arbitrary shapes; it needs sensible eps and min_samples.